# DATA OVERVIEW AND EDA

## 1: Project Overview
This notebook explores the processed Maji Ndogo agricultural dataset to understand its structure, assess data quality, examine the distribution of variables, and identify relationships between agricultural, environmental, and weather-related factors that influence crop productivity. The insights from this analysis will guide subsequent statistical analysis and predictive modeling.

## 2: Objectives

This notebook aims to:

- Understand the structure of the processed dataset.
- Assess overall data quality.
- Explore the distribution of numerical and categorical variables.
- Identify potential outliers and missing values.
- Investigate relationships between explanatory variables and crop yield.
- Generate insights that will guide statistical analysis and machine learning.

## 2: Import Libraries

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.pipeline import create_final_dataset

## 3: Load Dataset

In [ ]:
df = create_final_dataset()
df.head()

In [ ]:
def inspect_dataset(df):
    """
    This function provides a comprehensive overview of the dataset by displaying:
    - The first 5 rows (to understand column structure and data types)
    - The total number of rows and columns (to understand dataset size)
    - 5 random samples (to see data distribution without bias)
    _ The column structure and its data types/composition
    """
    return {
        'head': df.head(),
        'rows': df.shape[0],
        'columns': df.shape[1],
        'sample': df.sample(5, random_state=42),
        'info': df.info()
    }

inspection = inspect_dataset(df)

print(f"Rows: {inspection['rows']}")
print(f"Columns: {inspection['columns']}")
display(inspection['head'])
display(inspection['sample'])
display(inspection ['info'])

## 4: Data Quality Assessment

Before performing exploratory analysis, we evaluate the quality of the dataset by checking:

- Missing values
- Duplicate records
- Data types
- Invalid values
- Consistency of categorical variables
- Potential outliers

In [ ]:
def quality_assesment(df) -> pd.DataFrame |  None:
    """
    Print a quality assessment of the DataFrame.
    
    Args:
        df (pd.DataFrame): The DataFrame to assess
    returns pd.DataFrame
    """
  
    print("DATA QUALITY ASSESSMENT")
    
    missing_counts = df.isnull().sum()
    missing_values = missing_counts[missing_counts > 0]
    
    print("\n MISSING VALUES:")
    if len(missing_values) == 0:
        print("No missing values found!")
    else:
        for col, count in missing_values.items():
            pct = (count / len(df) * 100)
            print(f"  {col:20} | {count:5} missing ({pct:5.1f}%)")
    
    duplicate_count = df.duplicated().sum()
    
    print("\nDUPLICATE RECORDS:")
    if duplicate_count == 0:
        print("No duplicate rows found!")
    else:
        print(f"  Found {duplicate_count} duplicate rows ({duplicate_count/len(df)*100:.1f}%)")

    print("\n DATA TYPE COUNTS:")
    data_type_count = df.dtypes.value_counts()
    return data_type_count


quality_assesment(df)

In [ ]:
def get_categorical_columns(df) -> pd.DataFrame:
    columns = df.select_dtypes(str).columns.to_list()
    categorical_columns = pd.DataFrame(columns, columns=["categorical_columns"])
    return categorical_columns
get_categorical_columns(df)

In [ ]:
def list_categorical_values(df, column) -> pd.DataFrame:
    """
    This method lists unique values of categorical variables
    args:
        df: the Dataframe 
        column: the colums of the categorical variables
    returns: 
        pandas DataFrame: pd.DataFrame
    """
    unique_values = pd.DataFrame(df[column].unique(), columns=[column])
    return unique_values
list_categorical_values(df, "Location")

In [ ]:
list_categorical_values(df, "Soil_type")

In [ ]:
import pandas as pd
import numpy as np

def get_numeric_columns(df, exclude=None) -> list:
    """
    Gets all columns that are numeric except specified identifiers.
    
    Args:
        df: DataFrame
        exclude: list of column names to exclude (default: ["Weather_station_ID", "Field_ID"])
    
    Returns:
        list: List of numeric column names
    """
    if exclude is None:
        exclude = ["Weather_station_ID", "Field_ID"]
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols =[col for col in numeric_cols if col not in exclude]
    return numeric_cols

numeric_cols = get_numeric_columns(df)
display(numeric_cols)

In [ ]:
def summary_statistics(df, numeric_cols) -> pd.DataFrame:
    """
    Gets the descriptive/summary statistics of all numeric variables
    numeric unique identifiers
    
    args:
        df: The dataframe
        colums: The numeric variable column
    returns:
        pd.DataFrame of summarized statistics
    """

    numeric_stats = df.describe().T
    return numeric_stats
descriptive_num_stats = summary_statistics(df, numeric_cols)
descriptive_num_stats

## 5: Target Variable: Standard Yield

**Standard Yield** is our target variable because it's the outcome we want to predict. All other columns (weather, soil, field characteristics) serve as features that might influence yield.

In [ ]:
def descriptive_statistics(df, column)-> pd.DataFrame:
    desc_stats = pd.DataFrame(df[column].describe())
    return desc_stats
descriptive_statistics(df, "Standard_yield")

## Data Quality Findings

- The dataset contains 5,654 observations and no missing values.
- No duplicate records were identified.
- Numerical variables are correctly typed.
- Categorical variables contain consistent labels after preprocessing.
- Standard_yield contains no impossible negative values.

## 6: Univariate Analysis

This section explores individual variables to understand their distributions, variability, and potential anomalies. Numerical variables are examined using descriptive statistics and visualizations, while categorical variables are analyzed through frequency distributions.

### 1: Summary statistics

In [ ]:
summary_statistics(df, numeric_cols)

### 2: Distribution plots

In [ ]:
def plot_numerical_distribution(df, columns, title = None) -> None:
    """
    A Plot for the distribution of a numerical variables with counts.
    Args:
        df (DataFrame): The DataFrame containing the column
        column (str): The name of the numerical column to plot
        title (str, optional): Custom title for the plot
    Returns None but displays a visual plot.
    """
    
    mean = round(float(df[columns].mean()), 3)
    std = df[columns].std()

    fig, ax = plt.subplots(figsize = (8,6))
    sns.histplot(data = df[columns], bins=30, kde= True, ax = ax)
    ax.axvline(mean, color = "red", linewidth = 2, linestyle ="dashed", label = f"mean: {mean}")
    ax.axvline(mean + std, color = "orange", linewidth = 2, linestyle = "-", label = f"+1 std")
    ax.axvline(mean - std, color = "black", linewidth = 2, linestyle = "-", label = f"-1 std")

    ax.set_title(f"Distribution Of {columns}")
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_numerical_distribution(df, "Standard_yield")
plot_numerical_distribution(df, "Field_Rainfall")
plot_numerical_distribution(df, "Station_Rainfall")
plot_numerical_distribution(df, "Temperature")
plot_numerical_distribution(df, "Soil_fertility")

The histograms shows how the selected numerical variable is distributed around its mean, with the spread of values captured by the standard deviation lines. Most observations cluster near the center, while a longer tail or slight skewness suggests that a few values are unusually low or high. This indicates variability in the underlying feature and may point to the presence of outliers or influential observations

In [ ]:
def numerical_outlier_detection(df, columns, title = None) -> None:
    plt.Figure(figsize=(8, 6))
    sns.boxplot(data = df, x = df[columns])
    plt.title("Temperature Outliers")
    plt.show()
numerical_outlier_detection(df, "Standard_yield")
numerical_outlier_detection(df, "Field_Rainfall")
numerical_outlier_detection(df, "Station_Rainfall")
numerical_outlier_detection(df, "Temperature")
numerical_outlier_detection(df, "Soil_fertility")


## Part 2: Categorical Variables

In [ ]:
display(categorical_columns)

In [ ]:
def plot_categorical_distribution(df, column, title=None) -> None:
    """
    A Plot for the distribution of a categorical variables with counts and percentages.
    Args:
        df (DataFrame): The DataFrame containing the column
        column (str): The name of the categorical column to plot
        title (str, optional): Custom title for the plot
    """
    if title is None:
        title = f"{column} Distribution"

    plt.figure(figsize=(8, 5))
    ax = sns.countplot(
        data=df, 
        y=column, 
        order=df[column].value_counts().index)
    
    ax.bar_label( 
        ax.containers[0], 
        labels=[f'{val} ({val/len(df)*100:.1f}%)' for val in ax.containers[0].datavalues], 
        padding=5)
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Count", fontsize=12)
    plt.ylabel(column, fontsize=12)
    plt.tight_layout()
    plt.show()
    
plot_categorical_distribution(df, "Soil_type")
plot_categorical_distribution(df, "Crop_type")
plot_categorical_distribution(df, "Location")

The categorical distribution plots show that the dataset is unevenly distributed across categories, with some soil types, crop types, and locations appearing more frequently than others. This imbalance suggests that certain classes are better represented in the data, which may influence the reliability and generalizability of downstream analysis and modeling